# Phase 3 + 4 — Many-to-Many Pipeline (K=2 then K=3)

Run the two-phase M:M pipeline on all 9 ground-truth relation pairs.
Phase 3 uses `MAX_GROUP_SIZE=2`, Phase 4 uses `MAX_GROUP_SIZE=3`.
K=2 and K=3 runs produce different cache keys via `Parameters.max_group_size`.

In [1]:
# Cell 1 — environment setup (MUST run before any thesis-extension import)
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

_cwd = Path(os.getcwd()).resolve()
for _candidate in [_cwd, _cwd.parent, _cwd / "thesis-extension"]:
    if (_candidate / "pipeline.py").exists():
        _thesis_root = _candidate
        break
else:
    raise RuntimeError(
        f"Cannot locate thesis-extension root from CWD={_cwd}."
    )

_notebooks_dir = _thesis_root / "notebooks"
_project_root = _thesis_root.parent

os.environ.setdefault("RESULTS_DIR", str(_thesis_root / "results"))
os.environ.setdefault("TEMPLATE_DIR", str(_thesis_root / "templates"))

load_dotenv(_thesis_root / ".env", override=True)

for _d in [str(_thesis_root), str(_notebooks_dir)]:
    if _d not in sys.path:
        sys.path.insert(0, _d)

print("thesis-extension root:", _thesis_root)
print("RESULTS_DIR          :", os.environ["RESULTS_DIR"])

thesis-extension root: /Users/sadiazaman/Documents/my-improvement-project/thesis-extension
RESULTS_DIR          : /Users/sadiazaman/Documents/my-improvement-project/thesis-extension/results


In [2]:
# Cell 2 — imports and mock-mode check
import config as cfg_module
from config import config
from evaluation import evaluate_against_ground_truth
from models import Parameters, Side
from pipeline import schema_match
from utils import RELATION_PAIRS, load_relation

if not config["QUERY_LLM"]:
    print("WARNING: QUERY_LLM=False — running in mock mode. LLM will NOT be called.")

GT_PATH = str(_project_root / "updated_ground_truth.csv")
_model = (
    config["ANTHROPIC_MODEL"]
    if config["LLM_PROVIDER"] == "anthropic"
    else config["OPENAI_MODEL"]
)
print(f"Provider: {config['LLM_PROVIDER']}, Model: {_model}")

Provider: openai, Model: gpt-3.5-turbo


In [6]:
import nest_asyncio
nest_asyncio.apply()

In [7]:
# Cell 3 — load all 9 relation pairs
LOADED_PAIRS = []
for src_name, src_schema, tgt_name, tgt_schema in RELATION_PAIRS:
    src = load_relation(src_name, src_schema, Side.SOURCE)
    tgt = load_relation(tgt_name, tgt_schema, Side.TARGET)
    LOADED_PAIRS.append((src, tgt))
    print(
        f"  {src_name:20s} ({len(src.attributes):2d} attrs)"
        f" -> {tgt_name:22s} ({len(tgt.attributes):2d} attrs)"
    )

  Patients             ( 6 attrs) -> Person                 (18 attrs)
  Admissions           (16 attrs) -> Visit_Occurrence       (17 attrs)
  Admissions           (16 attrs) -> Death                  ( 7 attrs)
  Prescriptions        (17 attrs) -> Drug_Exposure          (23 attrs)
  Diagnoses_ICD        ( 8 attrs) -> Condition_Occurrence   (16 attrs)
  Transfers            ( 7 attrs) -> Care_Site              ( 6 attrs)
  Transfers            ( 7 attrs) -> Visit_Detail           (19 attrs)
  Admissions           (16 attrs) -> Visit_Detail           (19 attrs)
  Services             ( 5 attrs) -> Visit_Detail           (19 attrs)


In [8]:
# Cell 4 — Phase 3: K=2 run
# max_group_size=2 on Parameters is the cache key discriminator (via digest()).
# Also set config so pipeline.py uses the correct group size at runtime.
cfg_module.config["MAX_GROUP_SIZE"] = 2
cfg_module.config["PHASE_2_ENABLED"] = True
print(f"K=2 run — MAX_GROUP_SIZE={config['MAX_GROUP_SIZE']}, PHASE_2_ENABLED={config['PHASE_2_ENABLED']}")

results_k2 = []
for src, tgt in LOADED_PAIRS:
    params = Parameters(
        source_relation=src,
        target_relation=tgt,
        llm_model=_model,
        max_group_size=2,
    )
    result = schema_match(params)
    results_k2.append(result)
    print(
        f"  [K=2] {src.name}->{tgt.name}: "
        f"{len(result.pairs)} 1:1 pairs, "
        f"{len(result.group_pairs)} group pairs"
    )

Phase 1 generated 24 prompts for Patients->Person. With current n setting this means ~72 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.


K=2 run — MAX_GROUP_SIZE=2, PHASE_2_ENABLED=True
  [K=2] Patients->Person: 82 1:1 pairs, 0 group pairs


Phase 1 generated 33 prompts for Admissions->Visit_Occurrence. With current n setting this means ~99 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.
Answer 2 for prompt 76f84bee has invalid JSON
Traceback (most recent call last):
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_sending.py", line 120, in _send_one
    extract_json(answer)
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_sending.py", line 40, in extract_json
    start_decision = answer.answer.rindex("{")
                     ^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: substring not found
M:M JSON extraction failed for answer 1e86a60e
Traceback (most recent call last):
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_postprocessing.py", line 192, in postprocess_m2m_answers
    parsed = _extract_outermost_json(answer.answer)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sadiazaman/Documents/my-

  [K=2] Admissions->Visit_Occurrence: 140 1:1 pairs, 4 group pairs
  [K=2] Admissions->Death: 103 1:1 pairs, 0 group pairs


Phase 1 generated 40 prompts for Prescriptions->Drug_Exposure. With current n setting this means ~120 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.
Phase 1 generated 24 prompts for Diagnoses_ICD->Condition_Occurrence. With current n setting this means ~72 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.


...

  [K=2] Transfers->Care_Site: 42 1:1 pairs, 0 group pairs
  [K=2] Transfers->Visit_Detail: 83 1:1 pairs, 0 group pairs


Phase 1 generated 35 prompts for Admissions->Visit_Detail. With current n setting this means ~105 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.
M:M JSON extraction failed for answer 62b68914
Traceback (most recent call last):
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_postprocessing.py", line 192, in postprocess_m2m_answers
    parsed = _extract_outermost_json(answer.answer)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_postprocessing.py", line 58, in _extract_outermost_json
    return json.loads(raw)
           ^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python

  [K=2] Admissions->Visit_Detail: 207 1:1 pairs, 1 group pairs


Phase 1 generated 24 prompts for Services->Visit_Detail. With current n setting this means ~72 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.


  [K=2] Services->Visit_Detail: 72 1:1 pairs, 0 group pairs


In [9]:
# Cell 5 — Phase 4: K=3 run
cfg_module.config["MAX_GROUP_SIZE"] = 3
cfg_module.config["PHASE_2_ENABLED"] = True
print(f"K=3 run — MAX_GROUP_SIZE={config['MAX_GROUP_SIZE']}, PHASE_2_ENABLED={config['PHASE_2_ENABLED']}")

results_k3 = []
for src, tgt in LOADED_PAIRS:
    params = Parameters(
        source_relation=src,
        target_relation=tgt,
        llm_model=_model,
        max_group_size=3,
    )
    result = schema_match(params)
    results_k3.append(result)
    print(
        f"  [K=3] {src.name}->{tgt.name}: "
        f"{len(result.pairs)} 1:1 pairs, "
        f"{len(result.group_pairs)} group pairs"
    )

K=3 run — MAX_GROUP_SIZE=3, PHASE_2_ENABLED=True


Phase 1 generated 24 prompts for Patients->Person. With current n setting this means ~72 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.


  [K=3] Patients->Person: 91 1:1 pairs, 0 group pairs


Phase 1 generated 33 prompts for Admissions->Visit_Occurrence. With current n setting this means ~99 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.
Answer 0 for prompt a26fcca4 has invalid JSON
Traceback (most recent call last):
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_sending.py", line 120, in _send_one
    extract_json(answer)
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_sending.py", line 40, in extract_json
    start_decision = answer.answer.rindex("{")
                     ^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: substring not found
Skipping invalid answer (no parseable JSON): 01d327f0
Answer 0 for prompt f46d8269 has invalid JSON
Traceback (most recent call last):
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_sending.py", line 120, in _send_one
    extract_json(answer)
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_sen

  [K=3] Admissions->Visit_Occurrence: 137 1:1 pairs, 3 group pairs


...

  [K=3] Transfers->Care_Site: 40 1:1 pairs, 2 group pairs
  [K=3] Transfers->Visit_Detail: 91 1:1 pairs, 0 group pairs


Phase 1 generated 35 prompts for Admissions->Visit_Detail. With current n setting this means ~105 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.
Phase 1 generated 24 prompts for Services->Visit_Detail. With current n setting this means ~72 API calls. Consider reducing OPENAI_N or ANTHROPIC_N in .env.


  [K=3] Admissions->Visit_Detail: 197 1:1 pairs, 2 group pairs


Answer 2 for prompt 9bd41d43 has invalid JSON
Traceback (most recent call last):
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_sending.py", line 120, in _send_one
    extract_json(answer)
  File "/Users/sadiazaman/Documents/my-improvement-project/thesis-extension/prompt_sending.py", line 44, in extract_json
    return json.loads(raw_json)
           ^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/json/decoder

  [K=3] Services->Visit_Detail: 71 1:1 pairs, 2 group pairs


In [10]:
# Cell 6 — evaluate both runs and build comparison
import pandas as pd

def _eval_row(result, k: int) -> dict:
    report = evaluate_against_ground_truth(result, GT_PATH)
    src_n  = result.parameters.source_relation.name
    tgt_n  = result.parameters.target_relation.name
    return {
        "pair":           f"{src_n}->{tgt_n}",
        "k":              k,
        "f1_1to1":        round(report.f1_1to1, 3),
        "f1_group":       round(report.f1_group, 3),
        "group_pairs_found": len(result.group_pairs),
    }

rows_k2 = [_eval_row(r, 2) for r in results_k2]
rows_k3 = [_eval_row(r, 3) for r in results_k3]

df_k2 = pd.DataFrame(rows_k2)
df_k3 = pd.DataFrame(rows_k3)

# Side-by-side comparison
cmp = df_k2[["pair", "f1_1to1", "f1_group", "group_pairs_found"]].merge(
    df_k3[["pair", "f1_1to1", "f1_group", "group_pairs_found"]],
    on="pair",
    suffixes=("_k2", "_k3"),
)

print("=== K=2 vs K=3 Comparison ===")
print(cmp.to_string(index=False))
print(f"\nMean F1 (1:1) K=2: {df_k2['f1_1to1'].mean():.3f}  K=3: {df_k3['f1_1to1'].mean():.3f}")
print(f"Mean F1 (grp) K=2: {df_k2['f1_group'].mean():.3f}  K=3: {df_k3['f1_group'].mean():.3f}")
print(f"Group pairs   K=2: {df_k2['group_pairs_found'].sum()}  K=3: {df_k3['group_pairs_found'].sum()}")

=== K=2 vs K=3 Comparison ===
                               pair  f1_1to1_k2  f1_group_k2  group_pairs_found_k2  f1_1to1_k3  f1_group_k3  group_pairs_found_k3
                   Patients->Person       0.148          0.0                     0       0.154          0.0                     0
       Admissions->Visit_Occurrence       0.356          0.0                     4       0.361          0.0                     3
                  Admissions->Death       0.189          0.0                     0       0.204          0.0                     0
       Prescriptions->Drug_Exposure       0.118          0.0                     0       0.119          0.0                     0
Diagnoses_ICD->Condition_Occurrence       0.364          0.0                     1       0.392          0.0                     3
               Transfers->Care_Site       0.182          0.0                     0       0.200          0.0                     2
            Transfers->Visit_Detail       0.060          0.0

In [11]:
# Cell 7 — save results
import json

out_dir = Path(os.environ["RESULTS_DIR"])
out_dir.mkdir(parents=True, exist_ok=True)

k2_csv = out_dir / "m2m_k2_results.csv"
df_k2.to_csv(k2_csv, index=False)
print(f"Saved: {k2_csv}")

k3_csv = out_dir / "m2m_k3_results.csv"
df_k3.to_csv(k3_csv, index=False)
print(f"Saved: {k3_csv}")

cmp_csv = out_dir / "m2m_comparison.csv"
cmp.to_csv(cmp_csv, index=False)
print(f"Saved: {cmp_csv}")

Saved: /Users/sadiazaman/Documents/my-improvement-project/thesis-extension/results/m2m_k2_results.csv
Saved: /Users/sadiazaman/Documents/my-improvement-project/thesis-extension/results/m2m_k3_results.csv
Saved: /Users/sadiazaman/Documents/my-improvement-project/thesis-extension/results/m2m_comparison.csv


In [12]:
# Cell 8 — cost summary
cost_log = out_dir / "cost_log.jsonl"
if cost_log.exists():
    records = [
        json.loads(line)
        for line in cost_log.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    total_cost = sum(r.get("cost_usd", 0.0) for r in records)
    print(f"Total API calls     : {len(records)}")
    print(f"Estimated cost (USD): ${total_cost:.4f}")
else:
    print("No cost_log.jsonl found (mock mode or first run).")

Total API calls     : 2214
Estimated cost (USD): $1.2395
